In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

In [2]:
# ============================================================================
# IMPROVEMENT 1: Advanced Feature Engineering
# ============================================================================
def create_advanced_features(data):
    """Create sophisticated features from raw data"""
    df = data.copy()
    
    # Group features by type
    e_cols = [c for c in df.columns if c.startswith('E') and c != 'target']
    s_cols = [c for c in df.columns if c.startswith('S') and c != 'target']
    p_cols = [c for c in df.columns if c.startswith('P') and c != 'target']
    i_cols = [c for c in df.columns if c.startswith('I') and c != 'target']
    
    # Economic indicators features
    if e_cols:
        df['E_mean'] = df[e_cols].mean(axis=1)
        df['E_std'] = df[e_cols].std(axis=1)
        df['E_median'] = df[e_cols].median(axis=1)
        df['E_max'] = df[e_cols].max(axis=1)
        df['E_min'] = df[e_cols].min(axis=1)
        df['E_range'] = df['E_max'] - df['E_min']
        df['E_skew'] = df[e_cols].skew(axis=1)
        df['E_positive_ratio'] = (df[e_cols] > 0).sum(axis=1) / len(e_cols)
    
    # Sentiment indicators features
    if s_cols:
        df['S_mean'] = df[s_cols].mean(axis=1)
        df['S_std'] = df[s_cols].std(axis=1)
        df['S_median'] = df[s_cols].median(axis=1)
        df['S_max'] = df[s_cols].max(axis=1)
        df['S_min'] = df[s_cols].min(axis=1)
        df['S_range'] = df['S_max'] - df['S_min']
        df['S_positive_ratio'] = (df[s_cols] > 0).sum(axis=1) / len(s_cols)
    
    # Price indicators features
    if p_cols:
        df['P_mean'] = df[p_cols].mean(axis=1)
        df['P_std'] = df[p_cols].std(axis=1)
        df['P_median'] = df[p_cols].median(axis=1)
    
    # Interaction features (captures relationships)
    if e_cols and s_cols:
        df['ES_interaction'] = df['E_mean'] * df['S_mean']
        df['ES_ratio'] = df['E_mean'] / (df['S_mean'].abs() + 1e-5)
    
    if e_cols and p_cols:
        df['EP_interaction'] = df['E_mean'] * df['P_mean']
    
    if s_cols and p_cols:
        df['SP_interaction'] = df['S_mean'] * df['P_mean']
    
    # Overall volatility indicator
    all_numeric_cols = e_cols + s_cols + p_cols + i_cols
    if all_numeric_cols:
        df['overall_volatility'] = df[all_numeric_cols].std(axis=1)
        df['overall_mean'] = df[all_numeric_cols].mean(axis=1)
    
    return df

# ============================================================================
# IMPROVEMENT 2: Enhanced Preprocessing
# ============================================================================
def preprocessing(data, typ, impute_values=None):
    """Enhanced preprocessing with better imputation and feature engineering"""
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',  
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 
                     "I2", "P8", "P9", "P10", "P12", "P13"]
    
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else:
        data = data[main_features].copy()
    
    # Smart imputation with median (more robust than mean)
    if impute_values is None:
        impute_values = {}
        for col in data.columns:
            if col != 'target':
                impute_values[col] = data[col].median()
    
    for col in data.columns:
        if col != 'target':
            data[col].fillna(impute_values[col], inplace=True)
    
    # Drop rows with null targets only in training
    if 'target' in data.columns:
        data = data.dropna(subset=['target'])
    
    # Apply feature engineering
    data = create_advanced_features(data)
    
    if typ == "train":
        return data, impute_values
    else:
        return data

# ============================================================================
# IMPROVEMENT 3: Cross-Validation for Robust Evaluation
# ============================================================================
def cross_validate_model(X, y, model_params, n_splits=5):
    """Perform K-Fold cross-validation"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    cv_scores = []
    
    print("\n" + "="*60)
    print("Cross-Validation Results:")
    print("="*60)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train_fold = X.iloc[train_idx]
        y_train_fold = y.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_val_fold = y.iloc[val_idx]
        
        model = LGBMRegressor(**model_params)
        model.fit(
            X_train_fold, y_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            callbacks=[]
        )
        
        val_pred = model.predict(X_val_fold)
        mae = mean_absolute_error(y_val_fold, val_pred)
        rmse = np.sqrt(mean_squared_error(y_val_fold, val_pred))
        corr, _ = spearmanr(y_val_fold, val_pred)
        
        cv_scores.append(mae)
        print(f"Fold {fold + 1}: MAE={mae:.6f}, RMSE={rmse:.6f}, Corr={corr:.4f}")
    
    print(f"\nMean CV MAE: {np.mean(cv_scores):.6f} (+/- {np.std(cv_scores):.6f})")
    print("="*60)
    
    return cv_scores

# ============================================================================
# IMPROVEMENT 4: Adaptive Signal Calibration
# ============================================================================
def calibrate_signal_conversion(y_true, y_pred):
    """Calibrate signal conversion based on actual prediction distribution"""
    
    pred_mean = np.mean(y_pred)
    pred_std = np.std(y_pred)
    pred_min = np.min(y_pred)
    pred_max = np.max(y_pred)
    
    print("\n" + "="*60)
    print("Signal Calibration Analysis:")
    print("="*60)
    print(f"Prediction Statistics:")
    print(f"  Mean:  {pred_mean:.6f}")
    print(f"  Std:   {pred_std:.6f}")
    print(f"  Range: [{pred_min:.6f}, {pred_max:.6f}]")
    
    # Calculate optimal multiplier to map predictions to [0, 2]
    if pred_max != pred_min:
        multiplier = 2.0 / (pred_max - pred_min)
        offset = -pred_min * multiplier
    else:
        multiplier = 100.0
        offset = 1.0
    
    print(f"\nCalibrated Parameters:")
    print(f"  Multiplier: {multiplier:.2f}")
    print(f"  Offset:     {offset:.2f}")
    
    # Test the conversion
    test_signals = y_pred * multiplier + offset
    test_signals = np.clip(test_signals, 0, 2)
    
    print(f"\nSignal Distribution:")
    print(f"  Mean:  {np.mean(test_signals):.4f}")
    print(f"  Std:   {np.std(test_signals):.4f}")
    print(f"  Range: [{np.min(test_signals):.4f}, {np.max(test_signals):.4f}]")
    
    # Show signal categories
    bearish = np.sum(test_signals < 0.67) / len(test_signals) * 100
    neutral = np.sum((test_signals >= 0.67) & (test_signals < 1.33)) / len(test_signals) * 100
    bullish = np.sum(test_signals >= 1.33) / len(test_signals) * 100
    
    print(f"\nSignal Categories:")
    print(f"  Bearish (0.0-0.67):   {bearish:.1f}%")
    print(f"  Neutral (0.67-1.33):  {neutral:.1f}%")
    print(f"  Bullish (1.33-2.0):   {bullish:.1f}%")
    print("="*60)
    
    return multiplier, offset

# ============================================================================
# IMPROVEMENT 5: Feature Importance Analysis
# ============================================================================
def analyze_feature_importance(model, feature_names, top_n=20):
    """Analyze and display feature importance"""
    importances = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\n" + "="*60)
    print(f"Top {top_n} Most Important Features:")
    print("="*60)
    for idx, row in importances.head(top_n).iterrows():
        print(f"  {row['feature']:30} {row['importance']:10.2f}")
    print("="*60)
    
    return importances

# ============================================================================
# MAIN TRAINING PIPELINE
# ============================================================================

print("="*60)
print("HULL TACTICAL MARKET PREDICTION - LGBM MODEL")
print("="*60)

print("\nLoading data...")
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

print("Preprocessing data...")
train_processed, impute_values = preprocessing(train, "train")

print(f"\nDataset Information:")
print(f"  Shape: {train_processed.shape}")
print(f"  Features: {train_processed.shape[1] - 1}")
print(f"  Samples: {train_processed.shape[0]}")

# Split data
train_split, val_split = train_test_split(
    train_processed, test_size=0.15, random_state=42
)

X_train = train_split.drop(columns=["target"])
y_train = train_split['target']
X_val = val_split.drop(columns=["target"])
y_val = val_split['target']

print(f"\nData Split:")
print(f"  Training samples:   {len(X_train)}")
print(f"  Validation samples: {len(X_val)}")
print(f"\nTarget Statistics:")
print(f"  Mean: {y_train.mean():.6f}")
print(f"  Std:  {y_train.std():.6f}")
print(f"  Min:  {y_train.min():.6f}")
print(f"  Max:  {y_train.max():.6f}")

HULL TACTICAL MARKET PREDICTION - LGBM MODEL

Loading data...
Preprocessing data...

Dataset Information:
  Shape: (8990, 63)
  Features: 62
  Samples: 8990

Data Split:
  Training samples:   7641
  Validation samples: 1349

Target Statistics:
  Mean: 0.000481
  Std:  0.010558
  Min:  -0.039754
  Max:  0.040661


In [3]:
# Improved LGBM parameters (optimized for better generalization)
LGBM_params = {
    'boosting_type': 'gbdt', 
    'colsample_bytree': 0.9, 
    'learning_rate': 0.05,  # Lower for better generalization
    'max_bin': 77, 
    'max_depth': 8,  # Reduced to prevent overfitting
    'metric': 'mae', 
    'min_child_samples': 81, 
    'min_data_in_leaf': 21, 
    'n_estimators': 3000,  # Will use early stopping
    'num_leaves': 31,  # Reduced
    'objective': 'regression_l1', 
    'reg_alpha': 0.6, 
    'reg_lambda': 3.1, 
    'subsample': 0.73, 
    'verbosity': -1,
    'random_state': 42
}

# Optional: Run cross-validation (comment out if too slow)
print("\n" + "="*60)
print("Running Cross-Validation (Optional - comment out if slow)")
print("="*60)
# cv_scores = cross_validate_model(X_train, y_train, LGBM_params, n_splits=5)

# Train final model on full training set
print("\n" + "="*60)
print("Training Final LGBM Model...")
print("="*60)

LGBM_model = LGBMRegressor(**LGBM_params)
LGBM_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[]
)

# Evaluate on validation set
print("\n" + "="*60)
print("Validation Set Performance:")
print("="*60)

val_pred = LGBM_model.predict(X_val)
val_mae = mean_absolute_error(y_val, val_pred)
val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
val_corr, _ = spearmanr(y_val, val_pred)

print(f"MAE:         {val_mae:.6f}")
print(f"RMSE:        {val_rmse:.6f}")
print(f"Correlation: {val_corr:.4f}")
print(f"Pred Range:  [{val_pred.min():.6f}, {val_pred.max():.6f}]")

# Analyze feature importance
feature_importance = analyze_feature_importance(LGBM_model, X_train.columns, top_n=20)

# Calibrate signal conversion
SIGNAL_MULTIPLIER, SIGNAL_OFFSET = calibrate_signal_conversion(y_val, val_pred)

# Save model and parameters
print("\nSaving model and parameters...")
joblib.dump({
    'model': LGBM_model,
    'impute_values': impute_values,
    'signal_multiplier': SIGNAL_MULTIPLIER,
    'signal_offset': SIGNAL_OFFSET,
    'feature_names': X_train.columns.tolist()
}, 'lgbm_model.pkl')

print("✓ Model saved successfully as 'lgbm_model.pkl'")


Running Cross-Validation (Optional - comment out if slow)

Training Final LGBM Model...

Validation Set Performance:
MAE:         0.007791
RMSE:        0.010991
Correlation: 0.0365
Pred Range:  [-0.018382, 0.028167]

Top 20 Most Important Features:
  P13                               3027.00
  S12                               2970.00
  E19                               2861.00
  S5                                2652.00
  S9                                2571.00
  S10                               2380.00
  P_std                             2271.00
  P12                               2239.00
  S4                                2227.00
  S_median                          2130.00
  E4                                2097.00
  S6                                2096.00
  S2                                2081.00
  P_median                          1971.00
  S11                               1966.00
  E13                               1884.00
  overall_volatility                1817.00
  

In [4]:
# ============================================================================
# INFERENCE
# ============================================================================

# Load model and parameters
model_data = joblib.load('lgbm_model.pkl')
LGBM_model = model_data['model']
impute_values = model_data['impute_values']
SIGNAL_MULTIPLIER = model_data['signal_multiplier']
SIGNAL_OFFSET = model_data['signal_offset']

def predict(test: pl.DataFrame) -> float:
    """
    Main prediction function for Kaggle inference server
    
    Args:
        test: Polars DataFrame with single row of test data
        
    Returns:
        float: Signal value in range [0.0, 2.0]
               0.0-0.67: Bearish (low allocation)
               0.67-1.33: Neutral (moderate allocation)
               1.33-2.0: Bullish (high allocation)
    """
    try:
        # Convert polars to pandas
        test_df = test.to_pandas()
        
        # Preprocess with same parameters as training
        test_processed = preprocessing(test_df, 'inference', impute_values=impute_values)
        
        # Make prediction
        prediction = LGBM_model.predict(test_processed)
        
        # Handle array output
        if isinstance(prediction, np.ndarray):
            prediction = prediction[0]
        
        # Convert to signal in range [0, 2]
        signal = prediction * SIGNAL_MULTIPLIER + SIGNAL_OFFSET
        signal = np.clip(signal, 0.0, 2.0)
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        import traceback
        traceback.print_exc()
        return 1.0  # Return neutral signal on error

# Test prediction function
print("\n" + "="*60)
print("Testing Prediction Function:")
print("="*60)

test_sample = pl.from_pandas(val_split.head(5).drop(columns=['target']))
for i in range(min(5, len(test_sample))):
    sample = test_sample[i:i+1]
    signal = predict(sample)
    actual = y_val.iloc[i]
    print(f"Sample {i+1}: Signal={signal:.4f}, Actual Return={actual:.6f}")

print("\n✓ Prediction function tested successfully!")

# Initialize inference server
print("\n" + "="*60)
print("Initializing Inference Server...")
print("="*60)

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("Running in competition mode...")
    inference_server.serve()
else:
    print("Running in local test mode...")
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))


Testing Prediction Function:
Sample 1: Signal=0.7682, Actual Return=-0.001033
Sample 2: Signal=0.8796, Actual Return=-0.002057
Sample 3: Signal=1.0029, Actual Return=-0.011272
Sample 4: Signal=0.9221, Actual Return=0.006685
Sample 5: Signal=0.7059, Actual Return=-0.008755

✓ Prediction function tested successfully!

Initializing Inference Server...
Running in local test mode...
